# letsjam demo

A simple loop-track with 4 cars and 1 truck.

**Kernel**: select the `current` conda environment.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))

from letsjam import Map, Trajectory, TrafficWidget

# ── map ──────────────────────────────────────────────────────────────────────
m = Map(
    nodes=[
        (50,  50),   # 0 – top-left
        (350, 50),   # 1 – top-right
        (350, 350),  # 2 – bottom-right
        (50,  350),  # 3 – bottom-left
    ],
    streets=[
        (0, 1),  # 0 – top
        (1, 2),  # 1 – right
        (2, 3),  # 2 – bottom
        (3, 0),  # 3 – left
    ],
    car_length=12.0,
    truck_length=22.0,
)
m.add_park([(80, 80), (160, 80), (160, 160), (80, 160)])
m.add_river([(200, 50), (220, 200), (200, 350)], width=14)
m.add_cars(n_cars=4, n_trucks=1)

# ── simulation ───────────────────────────────────────────────────────────────
N_FRAMES = 120
SPEEDS   = [2.0, 1.6, 1.8, 2.2, 1.0]  # world-units per frame, one per vehicle

# initial offsets so vehicles don't all start at (0, 0)
street_len = [m.street_length(s) for s in range(4)]
total_loop  = sum(street_len)
offsets     = [i * total_loop / m.n_cars for i in range(m.n_cars)]
positions   = list(offsets)   # distance along full loop

def loop_pos(dist_along_loop):
    """Convert a scalar distance along the full loop to (street_id, dist)."""
    d = dist_along_loop % total_loop
    for s, sl in enumerate(street_len):
        if d < sl:
            return (s, d)
        d -= sl
    return (len(street_len) - 1, street_len[-1])

traj = Trajectory(m)
for _ in range(N_FRAMES):
    states = [loop_pos(positions[c]) for c in range(m.n_cars)]
    traj.append(states)
    for c in range(m.n_cars):
        positions[c] += SPEEDS[c]

# ── display ───────────────────────────────────────────────────────────────────
TrafficWidget.from_simulation(m, traj)